In [1]:
train_data_path = "../data/train_data.parquet"
station_coords_path = "station_coords.csv"
map_path = "../outputs/maps/zsr_graph.graphml"

In [2]:


import pandas as pd
import numpy as np
import json
import pyarrow.parquet as pq
from collections import defaultdict

parquet_file = pq.ParquetFile(train_data_path)

# name → list of GPS observations
name_lats = defaultdict(list)
name_lons = defaultdict(list)

for i in range(parquet_file.num_row_groups):
    df_chunk = parquet_file.read_row_group(i).to_pandas()
    if 'trains' not in df_chunk.columns:
        continue

    df_chunk['trains'] = df_chunk['trains'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    df = pd.json_normalize(
        df_chunk.explode('trains').reset_index(drop=True)['trains']
    )

    df["lat"] = df["Position"].apply(
        lambda p: p[0] if isinstance(p, list) and len(p) == 2 else np.nan
    )
    df["lon"] = df["Position"].apply(
        lambda p: p[1] if isinstance(p, list) and len(p) == 2 else np.nan
    )
    df["InfoZoStanice"] = df["InfoZoStanice"].astype(str).str.strip()

    # Only use snapshots where OrderCurrentStation matches a station stop
    # (i.e. train is AT a station, not between them — Position is reliable here)
    df["OrderCurrentStation"] = pd.to_numeric(
        df["OrderCurrentStation"], errors="coerce"
    )

    # Filter: train must be confirmed at station (PotvrdenyOdj gives clean events)
    at_station = df[
        df["InfoZoStanice"].notna() &
        (df["InfoZoStanice"] != "") &
        (df["InfoZoStanice"] != "nan") &
        df["lat"].notna() &
        df["lon"].notna()
    ]

    for _, row in at_station[["InfoZoStanice", "lat", "lon"]].iterrows():
        name_lats[row["InfoZoStanice"]].append(row["lat"])
        name_lons[row["InfoZoStanice"]].append(row["lon"])

    print(f"  chunk {i+1}/{parquet_file.num_row_groups} — "
          f"{len(name_lats)} station names with GPS so far")


  chunk 1/45 — 378 station names with GPS so far
  chunk 2/45 — 378 station names with GPS so far
  chunk 3/45 — 379 station names with GPS so far
  chunk 4/45 — 382 station names with GPS so far
  chunk 5/45 — 386 station names with GPS so far
  chunk 6/45 — 386 station names with GPS so far
  chunk 7/45 — 386 station names with GPS so far
  chunk 8/45 — 386 station names with GPS so far
  chunk 9/45 — 386 station names with GPS so far
  chunk 10/45 — 386 station names with GPS so far
  chunk 11/45 — 387 station names with GPS so far
  chunk 12/45 — 392 station names with GPS so far
  chunk 13/45 — 392 station names with GPS so far
  chunk 14/45 — 393 station names with GPS so far
  chunk 15/45 — 393 station names with GPS so far
  chunk 16/45 — 394 station names with GPS so far
  chunk 17/45 — 395 station names with GPS so far
  chunk 18/45 — 395 station names with GPS so far
  chunk 19/45 — 396 station names with GPS so far
  chunk 20/45 — 396 station names with GPS so far
  chunk 2

In [3]:

# ── Build clean name→GPS table ─────────────────────────────────────────────
records = []
for name in name_lats:
    lats = name_lats[name]
    lons = name_lons[name]

    # Use median — robust to outliers from train being mid-journey
    # when InfoZoStanice updates but Position hasn't caught up yet
    lat_med = np.median(lats)
    lon_med = np.median(lons)

    # IQR spread tells us if the GPS is consistent or noisy
    lat_spread = np.percentile(lats, 75) - np.percentile(lats, 25)
    lon_spread = np.percentile(lons, 75) - np.percentile(lons, 25)

    records.append({
        "name":       name,
        "lat":        round(lat_med, 6),
        "lon":        round(lon_med, 6),
        "n_samples":  len(lats),
        "lat_spread": round(lat_spread, 4),  # >0.01 means GPS is unreliable
        "lon_spread": round(lon_spread, 4),
    })

coords_df = pd.DataFrame(records).sort_values("n_samples", ascending=False)
coords_df.to_csv(station_coords_path, index=False)

In [4]:

# ── Diagnostics ────────────────────────────────────────────────────────────
print(f"\nTotal station names mapped: {len(coords_df)}")
print(f"High GPS spread (lat IQR > 0.01°, ~1km): "
      f"{(coords_df['lat_spread'] > 0.01).sum()}")
print()
print("=== Stations with unreliable GPS (check these manually) ===")
noisy = coords_df[coords_df['lat_spread'] > 0.01].sort_values("lat_spread", ascending=False)
print(noisy[["name","lat","lon","lat_spread","n_samples"]].head(20).to_string(index=False))
print()
print("=== Sample: top 20 by snapshot count ===")
print(coords_df.head(20)[["name","lat","lon","n_samples"]].to_string(index=False))


Total station names mapped: 416
High GPS spread (lat IQR > 0.01°, ~1km): 0

=== Stations with unreliable GPS (check these manually) ===
Empty DataFrame
Columns: [name, lat, lon, lat_spread, n_samples]
Index: []

=== Sample: top 20 by snapshot count ===
                     name       lat       lon  n_samples
Bratislava hlavná stanica 48.158900 17.106700     111835
                   Košice 48.722910 21.268720      79994
                   Trnava 48.369779 17.584820      62025
                    Čadca 49.445280 18.786810      55761
                   Žilina 49.227100 18.746040      50936
                   Púchov 49.113770 18.327170      48482
               Nové Zámky 47.995090 18.174130      42119
                     Kúty 48.661950 17.047530      41633
           Starý Smokovec 49.139298 20.222847      41523
                   Vrútky 49.115190 18.924420      41455
                    Kysak 48.852880 21.223930      40746
                  Galanta 48.185000 17.722970      40146
     

In [ ]:

import networkx as nx
import folium
import numpy as np

# Load your existing graph
G = nx.read_graphml(map_path)

# Load corrected coords
coords_df = pd.read_csv(station_coords_path)
name_to_coords = dict(zip(coords_df["name"], zip(coords_df["lat"], coords_df["lon"])))

# Patch nodes
patched = 0
still_missing = []
for node in G.nodes():
    coords = name_to_coords.get(node)
    if coords:
        G.nodes[node]["lat"] = coords[0]
        G.nodes[node]["lon"] = coords[1]
        patched += 1
    else:
        still_missing.append(node)

print(f"Patched: {patched}")
print(f"Still missing after patch: {len(still_missing)}")
if still_missing:
    print("Missing nodes (name mismatch):")
    for n in sorted(still_missing)[:30]:
        print(f"  '{n}'")

Patched: 385
Still missing after patch: 0


In [ ]:

import pandas as pd
import numpy as np
import json
import pyarrow.parquet as pq
import networkx as nx
import folium
from collections import defaultdict

parquet_file = pq.ParquetFile(train_data_path)

# Load the station mapping we already built
station_df = pd.read_csv(station_coords_path)
# name → (lat, lon)
name_to_coords = {
    row["name"]: (row["lat"], row["lon"])
    for _, row in station_df.iterrows()
    if pd.notna(row["lat"])
}

print(name_to_coords)

{'Bratislava hlavná stanica': (48.3019, 17.607965), 'Košice': (48.873332, 20.80377), 'Žilina': (49.11377, 18.718542), 'Zvolen osobná stanica': (48.56971, 19.14943), 'Banská Bystrica': (48.73475, 19.14921), 'Humenné': (48.93015, 21.82483), 'Nové Zámky': (48.08545, 18.17267), 'Prešov': (48.89062, 21.32044), 'Trnava': (48.369779, 17.5783), 'Kúty': (48.606372, 17.04753), 'Prievidza': (48.658867, 18.48969), 'Nitra': (48.35184, 18.07925), 'Čadca': (49.4036, 18.78681), 'Lučenec': (48.42328, 19.6675), 'Štrbské Pleso': (49.11853, 20.123146), 'Levice': (48.1854, 18.59264), 'Liptovský Mikuláš': (49.10815, 19.17215), 'Vrútky': (49.233573, 18.835748), 'Kúty št. hr.': (48.21356, 17.1067), 'Čadca št. hr.': (49.092, 19.41664), 'Komárno': (47.8615, 17.8518), 'Dunajská Streda': (48.03178, 17.452102), 'Veľký Horeš': (48.60604, 21.56711), 'Púchov': (48.93764, 18.113322), 'Bratislava-Nové Mesto': (48.3837, 17.80695), 'Galanta': (48.19587, 17.62928), 'Poprad-Tatry TEŽ': (49.118947, 20.222847), 'Margecany': 

In [2]:

# ── Accumulators ───────────────────────────────────────────────────────────
# Edge: (from_name, to_name) → list of delay values observed on that section
edge_delays = defaultdict(list)

# Track which train types use each edge (for tooltip)
edge_train_types = defaultdict(set)

for i in range(parquet_file.num_row_groups):
    df_chunk = parquet_file.read_row_group(i).to_pandas()

    if 'trains' not in df_chunk.columns:
        continue

    df_chunk['trains'] = df_chunk['trains'].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
    df = pd.json_normalize(df_chunk.explode('trains').reset_index(drop=True)['trains'])

    df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
    df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")
    df["date"]    = df["Cas_dt"].dt.date
    df["delay_gt"] = (df["Cas_dt"] - df["plan_dt"]).dt.total_seconds() / 60
    df["OrderCurrentStation"] = pd.to_numeric(df["OrderCurrentStation"], errors="coerce")

    # Clean station names
    df["InfoZoStanice"] = df["InfoZoStanice"].astype(str).str.strip()
    df = df[df["InfoZoStanice"].notna() & (df["InfoZoStanice"] != "") & (df["InfoZoStanice"] != "nan")]

    # Build edges from consecutive InfoZoStanice within each (train, date) run
    for (train, date), grp in df.groupby(["CisloVlaku", "date"]):
        g = (
            grp.sort_values("OrderCurrentStation")
            .drop_duplicates("OrderCurrentStation")
            .reset_index(drop=True)
        )
        if len(g) < 2:
            continue

        for idx in range(len(g) - 1):
            from_name  = g.iloc[idx]["InfoZoStanice"]
            to_name    = g.iloc[idx + 1]["InfoZoStanice"]
            delay      = g.iloc[idx + 1]["delay_gt"]
            train_type = g.iloc[idx].get("TypVlaku", "?")

            if from_name == to_name:
                continue  # same station repeated — skip

            edge_delays[(from_name, to_name)].append(delay)
            edge_train_types[(from_name, to_name)].add(str(train_type))

    print(f"  chunk {i+1}/{parquet_file.num_row_groups} — "
          f"{len(edge_delays)} unique sections found")
    
# Print info about edges:
print("\nSample edges with delay info:")
for idx, ((from_name, to_name), delays) in enumerate(edge_delays.items()):
    if idx >= 5:
        break
    train_types = edge_train_types[(from_name, to_name)]
    print(f"  {from_name} → {to_name}: "
          f"{len(delays)} delay records, "
          f"train types: {', '.join(train_types)}")

/tmp/ipykernel_950812/3136352410.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
/tmp/ipykernel_950812/3136352410.py:20: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")


  chunk 1/45 — 851 unique sections found


/tmp/ipykernel_950812/3136352410.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
/tmp/ipykernel_950812/3136352410.py:20: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")


  chunk 2/45 — 942 unique sections found


/tmp/ipykernel_950812/3136352410.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
/tmp/ipykernel_950812/3136352410.py:20: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")


...

  chunk 43/45 — 1667 unique sections found


/tmp/ipykernel_950812/3136352410.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
/tmp/ipykernel_950812/3136352410.py:20: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")


  chunk 44/45 — 1673 unique sections found


/tmp/ipykernel_950812/3136352410.py:19: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Cas_dt"]  = pd.to_datetime(df["Cas"],     dayfirst=True, errors="coerce")
/tmp/ipykernel_950812/3136352410.py:20: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["plan_dt"] = pd.to_datetime(df["CasPlan"], dayfirst=True, errors="coerce")


  chunk 45/45 — 1707 unique sections found

Sample edges with delay info:
  Čadca → Krásno nad Kysucou: 1305 delay records, train types: Os, RJ, SC, R, EN, EC
  Krásno nad Kysucou → Kysucké Nové Mesto: 1300 delay records, train types: Os, RJ, SC, R, EN, EC
  Kysucké Nové Mesto → Budatín odb.: 1301 delay records, train types: Os, RJ, SC, R, EN, EC
  Budatín odb. → Žilina: 1303 delay records, train types: Os, RJ, SC, R, EN, EC
  Žilina → Výhybňa Váh: 1756 delay records, train types: Os, RJ, SC, R, EN, Ex, EC


In [ ]:

# ── Build NetworkX DiGraph ─────────────────────────────────────────────────
G = nx.DiGraph()

for (from_name, to_name), delays in edge_delays.items():
    delays_clean = [d for d in delays if pd.notna(d) and -30 <= d <= 180]
    if len(delays_clean) < 5:
        continue  # skip edges with almost no data

    delay_arr = np.array(delays_clean)

    # Add nodes with GPS if we have them
    # Add nodes only with valid GPS data
    for name in (from_name, to_name):
        if name not in G.nodes:
            coords = name_to_coords.get(name)
            if coords and all(pd.notna(c) for c in coords):
                G.add_node(name, lat=float(coords[0]), lon=float(coords[1]))
            else:
                G.add_node(name) # Add node without lat/lon attributes

    G.add_edge(
        from_name, to_name,
        delay_mean   = round(float(np.mean(delay_arr)),   2),
        delay_median = round(float(np.median(delay_arr)), 2),
        delay_p90    = round(float(np.percentile(delay_arr, 90)), 2),
        delay_p95    = round(float(np.percentile(delay_arr, 95)), 2),
        delay_std    = round(float(np.std(delay_arr)),    2),
        delay_skew   = round(float(np.mean(delay_arr) - np.median(delay_arr)), 2),
        sample_count = len(delays_clean),
        train_types  = ", ".join(sorted(edge_train_types[(from_name, to_name)])),
    )

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# graph_path="zsr_graph.graphml"

# Save GraphML
nx.write_graphml(G, map_path)
print(f"Saved → {map_path}")


Graph: 385 nodes, 1018 edges
Saved → zsr_graph.graphml


In [8]:

# ── Build Folium HTML map ──────────────────────────────────────────────────
# Centre on Slovakia
m = folium.Map(location=[48.7, 19.5], zoom_start=8, tiles="CartoDB dark_matter")

# Colour edges by mean delay
max_delay = max((d["delay_mean"] for _, _, d in G.edges(data=True)), default=10)

def delay_color(mean_delay: float) -> str:
    if mean_delay <= 0:   return "#4CAF50"   # green  — on time / early
    if mean_delay <= 2:   return "#FFC107"   # yellow — slight delay
    if mean_delay <= 5:   return "#FF9800"   # orange — moderate
    return "#F44336"                          # red    — severe

# Draw edges first (so nodes render on top)
for from_name, to_name, data in G.edges(data=True):
    from_coords = (G.nodes[from_name].get("lat"), G.nodes[from_name].get("lon"))
    to_coords   = (G.nodes[to_name].get("lat"),   G.nodes[to_name].get("lon"))

    if None in from_coords or None in to_coords:
        continue
    if any(np.isnan(c) for c in from_coords + to_coords):
        continue

    tooltip = (
        f"<b>{from_name} → {to_name}</b><br>"
        f"Mean delay:   {data['delay_mean']} min<br>"
        f"Median delay: {data['delay_median']} min<br>"
        f"P90 delay:    {data['delay_p90']} min<br>"
        f"Std dev:      {data['delay_std']} min<br>"
        f"Skew:         {data['delay_skew']} min<br>"
        f"Samples:      {data['sample_count']}<br>"
        f"Train types:  {data['train_types']}"
    )

    # Weight line by sample count (busier sections = thicker)
    weight = min(1 + np.log1p(data["sample_count"]) / 3, 6)

    folium.PolyLine(
        locations=[from_coords, to_coords],
        color=delay_color(data["delay_mean"]),
        weight=weight,
        opacity=0.8,
        tooltip=folium.Tooltip(tooltip, sticky=True),
    ).add_to(m)

# Draw nodes
for name, data in G.nodes(data=True):
    lat, lon = data.get("lat"), data.get("lon")
    if lat is None or lon is None:
        continue
    if np.isnan(lat) or np.isnan(lon):
        continue

    degree = G.degree(name)
    radius = 4 + min(degree, 10)   # bigger circle = more connections

    # Collect node-level stats from its incoming edges
    in_delays = [G.edges[u, name]["delay_mean"] for u, _ in G.in_edges(name)]
    avg_in_delay = np.mean(in_delays) if in_delays else 0

    tooltip = (
        f"<b>{name}</b><br>"
        f"Connections: {degree}<br>"
        f"Avg incoming delay: {avg_in_delay:.1f} min<br>"
        f"Is junction: {'Yes' if degree >= 4 else 'No'}"
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color="white",
        fill=True,
        fill_color=delay_color(avg_in_delay),
        fill_opacity=0.9,
        tooltip=folium.Tooltip(tooltip, sticky=True),
    ).add_to(m)

    # Label major stations (high degree)
    if degree >= 4:
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(
                html=f'<div style="font-size:9px;color:white;white-space:nowrap;">{name}</div>',
                icon_size=(150, 20),
                icon_anchor=(0, 0),
            )
        ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:rgba(0,0,0,0.8);color:white;padding:12px;
            border-radius:8px;font-size:13px;">
  <b>Mean delay on section</b><br>
  <span style="color:#4CAF50">●</span> On time / early (≤ 0 min)<br>
  <span style="color:#FFC107">●</span> Slight (0–2 min)<br>
  <span style="color:#FF9800">●</span> Moderate (2–5 min)<br>
  <span style="color:#F44336">●</span> Severe (&gt; 5 min)<br>
  <br><b>Line thickness</b> = traffic volume
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("zsr_network_map.html")
print("Saved → zsr_network_map.html")


Saved → zsr_network_map.html


In [9]:

# ── Quick graph diagnostics ────────────────────────────────────────────────
print(f"\n=== Graph diagnostics ===")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

components = list(nx.weakly_connected_components(G))
print(f"Weakly connected components: {len(components)}")
print(f"Largest component: {max(len(c) for c in components)} nodes")

top_nodes = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop 10 hubs:")
for name, deg in top_nodes:
    print(f"  {name:<35} degree={deg}")

top_edges = sorted(
    G.edges(data=True),
    key=lambda x: x[2]["delay_mean"],
    reverse=True
)[:10]
print(f"\nTop 10 highest delay sections:")
for u, v, d in top_edges:
    print(f"  {u} → {v}:  mean={d['delay_mean']} min  samples={d['sample_count']}")


=== Graph diagnostics ===
Nodes: 385
Edges: 1018
Weakly connected components: 2
Largest component: 376 nodes

Top 10 hubs:
  Bratislava hlavná stanica           degree=17
  Odb. Vinohrady                      degree=13
  Košice                              degree=12
  Nové Mesto nad Váhom                degree=12
  Bratislava-Nové Mesto               degree=12
  Šurany                              degree=12
  Moldava nad Bodvou                  degree=12
  Kysak                               degree=11
  Trnava                              degree=11
  Pezinok                             degree=11

Top 10 highest delay sections:
  Topoľčany → Chynorany:  mean=32.2 min  samples=5
  Gbelce → Štúrovo tranzitná skupina:  mean=20.6 min  samples=10
  Čierna nad Tisou → Čierna nad Tisou št. hr.:  mean=18.94 min  samples=96
  Trnava → Veľké Kostoľany:  mean=18.4 min  samples=5
  Fiľakovo → Lučenec:  mean=17.48 min  samples=23
  Nové Zámky → Šurany:  mean=16.83 min  samples=12
  Fiľakovo → Výh. 

In [1]:
no_coords = [n for n, d in G.nodes(data=True) 
             if d.get("lat") is None or np.isnan(d.get("lat", np.nan))]
print(no_coords)
print(len(no_coords))

NameError: name 'G' is not defined